# Libraries Import and Base Path initialization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics

In [ ]:
import boxmot.trackers.ocsort.ocsort as o
print(dir(o))

from boxmot import Boxmot
help(Boxmot)

from boxmot.trackers.ocsort.ocsort import OcSort
help(OcSort)

## Osnet location

Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF

To: /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/models/osnet_x0_25_msmt17.pt

100%|██████████| 3.06M/3.06M [00:00<00:00, 15.4MB/s]
SUCCESS  | Loaded pretrained weights from /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/models/osnet_x0_25_msmt17.pt

In [1]:
import os, cv2, json, math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import tensorflow as tf
import tensorflow_hub as hub
from ultralytics import YOLO
from boxmot.trackers.ocsort.ocsort import OcSort
# from boxmot.trackers.deepocsort.deepocsort import DeepOcSort
# import pytorch as torch

# from mmengine.config import Config
# from mmengine.registry import MODELS
# from mmengine.runner import load_checkpoint
# from mmaction.apis import init_recognizer

# # This function will now work correctly because we are running from the cloned directory
# from mmaction.utils import register_all_modules
# register_all_modules(init_default_scope=True) # We set the scope manually later

#Colab Base Path
# base_path = "/content/drive/MyDrive/SMT 6/CV/UAS"

#Local Base Path
base_path = ""

# === Dataset Paths ===
data_path = os.path.join(base_path, "match_videos")
# data_path = os.path.join(base_path, "practice_videos")

video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) TWT 2024.mp4")
# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) 1 round.mp4")
# video_path = os.path.join(data_path, "lowhigh(bryan) vs ninjakilla(law)_1round.mp4")
# video_path = os.path.join(data_path, "Bryan_L_combo_2.mp4")

# annotation_path = os.path.join(base_path, "match_videos/Knee(Bryan) vs Double(Law) TWT 2024.json")
annotation_path = os.path.join(data_path, "Knee_reindexed.json")

output_dir = os.path.join(data_path, "frames")
kp_dir = os.path.join(data_path)

# video_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_trimmed.mp4")
# annotation_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_2.json")
# output_dir = os.path.join(base_path, "Bryan_2/frames")

# === Load movenet and YOLO models ===
movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4").signatures['serving_default']
# yolo = YOLO("yolo11s.pt")
# yolo = YOLO("yolo11m.pt")
# yolo = YOLO("yolo26s.pt")
yolo  = YOLO("runs/detect/twt/weights/best.pt")
yolo.to("mps")


# os.makedirs(output_dir, exist_ok=True)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_runnin

# YOLO and ByteTrack character tracking

In [2]:
tracker = OcSort(
    half=True,
    device="mps",
    # reid_weights="osnet_x0_25_msmt17.pt",
    max_age=90,
    min_hits=2,
    iou_threshold=0.15,
    det_thresh=0.20
)

def yolo_detect(image):
    result = yolo(image, conf=0.35, iou=0.3, classes=[0])[0]

    # the results of yolo.predict contains list of object per frame it detects, for example image will have 1 object in the list
    # while video will have as many object in it as the video frames
    # we will access the first object as it is an image
    # Play with conf(minimum conf to be detected) and 
    # iou (how much the boxes can overlap to be considered the same object)
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 6), dtype=np.float32)

    xyxy = result.boxes.xyxy.cpu().numpy() # convert boxes x1, y1, x2, y2 of selected object to numpy, then to int
    conf = result.boxes.conf.cpu().numpy().reshape(-1, 1) # convert boxes confidence of selected object to numpy
    cls = result.boxes.cls.cpu().numpy().reshape(-1, 1) # convert boxes class of selected object to numpy

    detections = np.hstack((xyxy, conf, cls))
    # stack boxes, conf, cls horizontally (it only accepts tuple so we encapsulate it with double ()

    # print("Results xyxy", result.boxes.xyxy) # print the x1, y1, x2, y2 from boxes of the object
    # print("detections", detections, type(detections))
    return detections

def ocsort_tracking(detections, image): 
    if detections.shape[0] == 0:
        return None
    
    tracks = tracker.update(detections, image)
    if tracks is None or len(tracks) == 0:
        return None
    
    ids = tracks[:, 4].astype(int).reshape(-1, 1)
    boxes = tracks[:, :4].astype(int)
    id_box_array = np.hstack((ids, boxes))
    return id_box_array

def pad_box(h, w, box, padding=25):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    return np.array([_, x1, y1, x2, y2], dtype=int)
    
def crop_roi(frame, box):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    # print("x1: ", x1, "y1: ", y1, "x2: ", x2, "y2: ", y2)
    return frame[y1:y2, x1:x2]

def display_roi(player1_roi, player2_roi):
    fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (4, 8))
    axes[0].imshow(player1_roi)
    axes[1].imshow(player2_roi)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def is_timer_missing(frame, edge_threshold=50):
    """
    Checks the Tekken timer UI using Edge Detection.
    Returns True if the sharp metallic borders of the numbers are missing.
    """
    y1, y2 = 26, 86
    x1, x2 = 600, 680
    timer_roi = frame[y1:y2, x1:x2]
    
    gray = cv2.cvtColor(timer_roi, cv2.COLOR_BGR2GRAY)
    
    # cv2.Canny highlights sharp transitions. 
    # The silver border of the font will light up brilliantly here.
    edges = cv2.Canny(gray, 100, 200)
    
    # Count how many 'edge' pixels exist in that small box
    edge_count = cv2.countNonZero(edges)
    
    # If the count drops below the threshold, the timer is gone.
    return edge_count < edge_threshold

def is_cinematic_zoom(boxes, frame_height, threshold=0.75): # Raised to 85%
    """
    Detects if the camera has zoomed in brutally for a Rage Art, Tornado, or K.O.
    """
    if boxes is None or len(boxes) == 0:
        return False
        
    for box in boxes:
        box_h = box[3] - box[1] 
        if box_h > (frame_height * threshold):
            return True
            
    return False

WARNING  | Max age > max observations, increasing size of max observations...
SUCCESS  | OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False, asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


# Movenet Holistic Function

In [3]:
def run_movenet(image, input_size=192):
    image = tf.image.resize_with_pad(image, input_size, input_size)
    # image = tf.cast(image, dtype=tf.float32) # if using gpu ?? not sure
    image = tf.cast(image, dtype=tf.int32) # if using cpu
    image = tf.expand_dims(image, axis=0)

    output = movenet(image)
    # The output is a dictionary, and we need the keypoints from 'output_0'
    keypoints_list = output['output_0'].numpy()[0]
    # print("Image shape", image.shape)
    # print("Keypoints list shape", keypoints_list.shape)
    # print("Keypoints list [0] shape", keypoints_list[0].shape)

    return keypoints_list[0]

def denormalize_points(points, original_height, original_width, input_size=192):
    """
    Converts normalized keypoints or bounding boxes from MoveNet output
    back to original image coordinates.
    """
    scale = min(input_size / original_height, input_size / original_width)
    new_height = original_height * scale
    new_width = original_width * scale
    pad_y = (input_size - new_height) / 2
    pad_x = (input_size - new_width) / 2

    y, x, c = points
    x_abs = ((x * input_size) - pad_x) / scale
    y_abs = ((y * input_size) - pad_y) / scale
    return (int(y_abs), int(x_abs), c)

def normalize_points_to_full_frame(kp_array, box, full_height, full_width):
    _, x1, y1, x2, y2 = box
    roi_height = y2-y1
    roi_width = x2-x1

    kp_full = []
    for kp in kp_array:
        y_roi, x_roi, c = denormalize_points(kp, roi_height, roi_width, 192)
        y_full = (y_roi + y1) / full_height
        x_full = (x_roi + x1) / full_width
        kp_full.append([y_full, x_full, c])

    return np.array(kp_full)
        

def interpolate_points(player_kp):
    player_kp = np.array(player_kp)
    for kp in range(player_kp.shape[1]):
        for coord in range(player_kp.shape[2]):
            data = player_kp[:, kp, coord]
            nans = np.isnan(data) # nans mask example: [true, false, true] based on the positions
            if np.any(~nans):
                data[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), data[~nans])
            player_kp[:, kp, coord] = data
            print(data)
    return player_kp
# for kp in player1_kp:
#     print(kp)
#     y, x, c = denormalize_points(kp, original_height, original_width)

# Movenet Holistic Extraction

In [ ]:
input_size = 192
cap = cv2.VideoCapture(video_path)
print(video_path, os.path.exists(video_path))

player1_kp = []
player2_kp = []
player_set = False
player1_id, player2_id = None, None
player1_box, player2_box, other_box = None, None, None
player1_kp_full, player2_kp_full, other_kp_full = None, None, None
frame_count = 0

missing_timer_frames = 0
buffer_limit = 10  # 10 frames ignores brief juggles, but catches the 15-frame practice reset
tracking_active = True

cinematic_frames = 0
cinematic_buffer = 5 # Wait 5 frames to confirm a zoom

while cap.isOpened(): # read every single frame of the video
    ret, frame_bgr = cap.read()
    if not ret:
        print("End of video")
        break

    frame_height = frame_bgr.shape[0]

    # --- 1. CHECK THE TIMER ---
    if is_timer_missing(frame_bgr):
        missing_timer_frames += 1
    else:
        missing_timer_frames = 0
        tracking_active = True # Timer is clearly visible, tracking is safe

    # --- HANDLE THE RESET STATE ---
    if missing_timer_frames > buffer_limit:
        
        # Only print and wipe memory the FIRST time we cross the threshold
        if tracking_active:
            print(f"Scene transition detected at frame {frame_count}. Pausing tracking...")
            tracking_active = False 
            
            # Wipe your tracker memory here!
            last_p1_box = None
            last_p2_box = None
            # If using an OC-SORT instance, destroy/re-init it here.
            player_set = False

        # We are in a reset state (black screen or waiting for fade-in).
        # Append empty frames to keep your temporal arrays perfectly aligned for the pipeline.
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        
        frame_count += 1
        
        # Skip the rest of the loop entirely. Do not run YOLO.
        continue


    # --- 2. THE CINEMATIC CHECK ---
    # Run your raw YOLO detection ONCE per frame
    detections = yolo_detect(frame_bgr) 
    
    # # Check if the boxes are massive (camera zoomed in)
    # if is_cinematic_zoom(detections, frame_height, threshold=0.85):
    #     cinematic_frames += 1
    # else:
    #     cinematic_frames = 0

    # if cinematic_frames > cinematic_buffer:
    #     print(f"Cinematic zoom detected at frame {frame_count}. Pausing tracking...")
        
    #     # Append NaNs because no actual gameplay is happening
    #     player1_kp.append(np.full((17, 3), np.nan))
    #     player2_kp.append(np.full((17, 3), np.nan))
        
    #     # Wipe the player memory! 
    #     # Characters often land in different spots after a Tornado/Rage Art.
    #     # This forces the logic to re-evaluate who is on the left/right when zooming out.
    #     player_set = False 
        
    #     frame_count += 1
    #     continue # Skip OC-SORT and MoveNet

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    id_box_array = ocsort_tracking(detections, frame_bgr) # 2d array containing id_box from p1 and 2
    # print(id_box_array)

    if id_box_array is None or id_box_array.size == 0:
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        frame_count += 1
        continue

    # if frame_count == 0:
    #     original_height, original_width = frame_bgr.shape[:2]
    #     print("Original height, original_width", original_height, original_width)
    #     player1_id = id_box_array[0, 0]
    #     player2_id = id_box_array[1, 0]

    if id_box_array is not None and id_box_array.shape[0] >= 2 and not player_set:
        original_height, original_width = frame_bgr.shape[:2]
        centers_x = (id_box_array[:,1] + id_box_array[:,3]) / 2
        order = np.argsort(centers_x)           # left -> right
        player1_id = int(id_box_array[order[0], 0])
        player2_id = int(id_box_array[order[1], 0])
        player_set = True

    p1_exist = any(id_box_array[:, 0] == player1_id)
    p2_exist = any(id_box_array[:, 0] == player2_id)

    other_mask = ~np.isin(id_box_array[:, 0], [player1_id, player2_id])
    has_other = np.any(other_mask)
    
    if p1_exist:
        player1_box = id_box_array[id_box_array[:, 0] == player1_id][0]
        # print("p1 box", player1_box)
        player1_box_padded = pad_box(original_height, original_width, player1_box)
        player1_roi = crop_roi(frame_bgr, player1_box_padded)
        player1_kp_raw = run_movenet(player1_roi, input_size)
        player1_kp_full = normalize_points_to_full_frame(player1_kp_raw, player1_box_padded, original_height, original_width)
        player1_kp.append(player1_kp_full)
    else:
        player1_box = None
        player1_box_padded = None
        player1_kp_full = None
        player1_kp.append(np.full((17, 3), np.nan))

    if p2_exist:
        player2_box = id_box_array[id_box_array[:, 0] == player2_id][0]
        player2_box_padded = pad_box(original_height, original_width, player2_box)
        player2_roi = crop_roi(frame_bgr, player2_box_padded)
        player2_kp_raw = run_movenet(player2_roi, input_size)
        player2_kp_full = normalize_points_to_full_frame(player2_kp_raw, player2_box_padded, original_height, original_width)
        player2_kp.append(player2_kp_full)
    else:
        player2_box = None
        player2_box_padded = None
        player2_kp_full = None
        player2_kp.append(np.full((17, 3), np.nan))

    if has_other:
        other_box = id_box_array[other_mask][0]
        other_box_padded = pad_box(original_height, original_width, other_box)
        other_roi = crop_roi(frame_bgr, other_box_padded)
        other_kp_raw = run_movenet(other_roi, input_size)
        other_kp_full = normalize_points_to_full_frame(other_kp_raw, other_box, original_height, original_width)


    # Visualization: draw player 1 (green) and player 2 (red)
    color_p1 = (0, 255, 0)  # green (B, G, R)
    color_p2 = (0, 0, 255)  # red
    
    # Boxes + IDs
    if player1_box is not None:
        obj_id, x1, y1, x2, y2 = player1_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p1, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p1, thickness=1, lineType=cv2.LINE_AA)
    
    if player2_box is not None:
        obj_id, x1, y1, x2, y2 = player2_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p2, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    if other_box is not None:
        obj_id, x1, y1, x2, y2 = other_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (255, 0, 0), thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    # Keypoints
    if player1_kp_full is not None:
        for kp in player1_kp_full:
            y, x, c = kp
            y, x = y * original_height, x * original_width
            cv2.circle(frame_bgr, (int(x), int(y)), 3, color_p1, thickness=2, lineType=cv2.LINE_AA)
    
    if player2_kp_full is not None:
        for kp in player2_kp_full:
            y, x, c = kp
            y, x = y * original_height, x * original_width
            cv2.circle(frame_bgr, (int(x), int(y)), 3, color_p2, thickness=2, lineType=cv2.LINE_AA)

    cv2.imshow('Process feed', frame_bgr)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    # plt.imshow(frame_rgb)
    # plt.show()

    frame_count += 1

cap.release()

player1_kp = interpolate_points(player1_kp)
player2_kp = interpolate_points(player2_kp)

np.save(os.path.join(kp_dir, "player1_kp"), player1_kp)
np.save(os.path.join(kp_dir, "player2_kp"), player2_kp)

Sure! Here's a concise summary of everything we discussed, formatted in markdown for easy reference:

---

## 📚 Summary: MMAction2 Skeleton Dataset Format & Preparation Steps

Skeleton-based Action Recognition in MMAction2 doesn’t require splitting the original video, but it **does require splitting the keypoint data** into action-based segments.

---

### 🧬 Dataset Format Overview (`.pkl`)

```python
{
  "split": {
    "train": ["clip1", "clip2", ...],
    "val": ["clip7", "clip8", ...],
    ...
  },
  "annotations": [
    {
      "frame_dir": "clip1",
      "label": 0,
      "img_shape": (1080, 1920),
      "original_shape": (1080, 1920),
      "total_frames": 87,
      "keypoint": np.ndarray([M, T, V, C]),
      "keypoint_score": np.ndarray([M, T, V])
    },
    ...
  ]
}
```

- **`frame_dir`**: Unique name for each clip
- **`label`**: Action class (int)
- **`img_shape` & `original_shape`**: Optional frame resolution
- **`total_frames`**: Frames in the segment
- **`keypoint`**: Shape `[M x T x V x C]` (people, frames, joints, coords)
- **`keypoint_score`**: Confidence for each keypoint `[M x T x V]`

---

### ⚙️ Steps to Prepare from a Long Video

If you already extracted full video keypoints:

1. **Use Annotations**  
   Get frame ranges for each action from your annotation file.

2. **Slice Keypoint Arrays**  
   Extract each action clip from the full keypoint array using its frame indices.

3. **Assign Clip Identifiers**  
   Name each segment like `clip001`, `clip002`, etc.

4. **Group into Splits**  
   Organize clip names into `'train'`, `'val'`, etc. inside the `split` dictionary.

5. **Build Annotations List**  
   For each clip, create a dictionary with all required fields and add it to `annotations`.

6. **Save to Pickle**  
   Combine `split` and `annotations` into a Python dict and save as `.pkl`.

---

Want me to build a sample Python script to help automate these steps? Happy to dive in! 💻

# Prepare dataloader

In [22]:
# JSON Annotation
with open(annotation_path) as f:
    annotations = json.load(f)

# Keypoints
player1_kp = np.load(os.path.join(kp_dir, "player1_kp.npy"))
player2_kp = np.load(os.path.join(kp_dir, "player2_kp.npy"))

# sequences_id = list(annotations.keys())[0]
# data = annotations[sequences_id]

# print(sequences_id, data)
# print(annotations.keys())
# print(player1_kp[0])

for sequence_id, data in annotations.items():
    print(sequence_id)
    print(data)
    player = data["player"]
    start_frame = data["start_frame"]
    end_frame = data["end_frame"]

    if player == "player1":
        kp = player1_kp[start_frame:end_frame+1]
        print(len(kp))

    print(start_frame, end_frame)

p1_sequence_1
{'player': 'player1', 'character': 'Bryan', 'move': 'db+3', 'start_frame': 30, 'end_frame': 44, 'start_time': 0.5, 'end_time': 0.7333333333333333, 'duration_frames': 14, 'annotation_type': 'single_character'}
15
30 44
p2_sequence_2
{'player': 'player2', 'character': 'Law', 'move': '1', 'start_frame': 87, 'end_frame': 92, 'start_time': 1.45, 'end_time': 1.5333333333333334, 'duration_frames': 5, 'annotation_type': 'single_character'}
87 92
p2_sequence_3
{'player': 'player2', 'character': 'Law', 'move': 'f+1+2', 'start_frame': 124, 'end_frame': 140, 'start_time': 2.066666666666667, 'end_time': 2.3333333333333335, 'duration_frames': 16, 'annotation_type': 'single_character'}
124 140
p2_sequence_4
{'player': 'player2', 'character': 'Law', 'move': 'df+1', 'start_frame': 179, 'end_frame': 191, 'start_time': 2.9833333333333334, 'end_time': 3.183333333333333, 'duration_frames': 12, 'annotation_type': 'single_character'}
179 191
p2_sequence_5
{'player': 'player2', 'character': 'Law

# Prepare STGCN++ model

In [ ]:
config_file = "https://github.com/open-mmlab/mmaction2/blob/main/configs/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d.py"
checkpoint = "https://download.openmmlab.com/mmaction/v1.0/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d_20221228-19a34aba.pth"